# Week 3 — Cleaning your data
## group3b · School results

**Your question**
> Which subjects show the weakest results, and how does attendance relate to score across terms?

**What this notebook does.** Pulls the raw data out of the database, fixes the
problems you found in week 2, and writes clean tables back into your own
schema. Power BI reads those tables in week 4.

**How to use it.** Every section has an explanation, then a cell to run, then
a `TODO` where you make a decision. The decisions are the work — the code
around them is scaffolding so you are not starting from a blank page.

**Before you start:** have your week 2 `data_quality_notes.md` open. Every
number you wrote there tells you what to fix here.

---
### One rule
Run this notebook top to bottom, in order. If it only works when you run cells
out of sequence, it is not finished — someone else in your group has to be able
to run it from scratch and get the same tables.


## 1. Setup

Run this once per session. Colab forgets everything when it disconnects, so
you will run it again tomorrow.


In [1]:
!pip install -q psycopg2-binary sqlalchemy

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from getpass import getpass

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
print("pandas", pd.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 23.0 MB/s eta 0:00:00
pandas 2.2.3


### Connect

`getpass` hides your password as you type it, so it never ends up saved in the
notebook. **Never type your password directly into a cell** — the notebook goes
to GitHub and the password would go with it.


In [8]:
HOST   = "internship-db.coh86gwewtxb.us-east-1.rds.amazonaws.com"
DB     = "internship"
USER   = "group3b"
SCHEMA_RAW   = "raw_school"
SCHEMA_MINE  = "group3b"

password = getpass("Password for group3b: ")

engine = create_engine(
    f"postgresql+psycopg2://{USER}:{password}@{HOST}:5432/{DB}?sslmode=require"
)

# quick check
pd.read_sql(f"SELECT count(*) AS rows FROM {SCHEMA_RAW}.results", engine)


Password for group3b: ··········


,rows
0,30120


You should see **30,120**. If not, stop — something is
wrong with the connection, not with your code.


## 2. Load the raw tables

Pull all four into pandas. They are small enough to hold in memory
comfortably.


In [9]:
df = pd.read_sql(f"SELECT * FROM {SCHEMA_RAW}.results", engine)
students = pd.read_sql(f"SELECT * FROM {SCHEMA_RAW}.students", engine)
subjects = pd.read_sql(f"SELECT * FROM {SCHEMA_RAW}.subjects", engine)
teachers = pd.read_sql(f"SELECT * FROM {SCHEMA_RAW}.teachers", engine)

print('results   ', df.shape)
print('students'.ljust(12), students.shape)
print('subjects'.ljust(12), subjects.shape)
print('teachers'.ljust(12), teachers.shape)


results    (30120, 8)
students     (1200, 6)
subjects     (10, 3)
teachers     (50, 4)


`.shape` gives (rows, columns). Check these against what you
recorded in week 2 — if a number is different, find out why before going on.


In [10]:
df.head(10)


,result_id,exam_date,student_id,subject_id,teacher_id,term,score,attendance_pct
0,5286,2024-01-05,633,9,44,Term 1,NaN,75.1
1,11177,18/05/2025,421,9,50,Term 2,57.1,94.4
2,10157,2024-05-04,222,3,22,Term 3,48.9,83.2
3,16448,2025-06-21,961,10,42,Term 3,41.5,94.4
4,29013,2025-09-29,190,9,23,Term 3,27.2,94.4
5,767,2025-07-19,730,10,13,Term 1,57.4,96.3
6,21201,2024-10-03,133,2,45,Term 1,68.8,79.0
7,2321,2024-03-09,798,10,34,Term 3,5.8,76.5
8,15217,2024-12-21,209,2,45,Term 3,63.9,90.3
9,22951,2025-05-04,338,4,44,Term 3,53.7,94.5


In [11]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30120 entries, 0 to 30119
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   result_id       30120 non-null  int64  
 1   exam_date       30120 non-null  object 
 2   student_id      30120 non-null  int64  
 3   subject_id      30120 non-null  int64  
 4   teacher_id      30120 non-null  int64  
 5   term            30120 non-null  object 
 6   score           28613 non-null  float64
 7   attendance_pct  30120 non-null  float64
dtypes: float64(2), int64(4), object(2)
memory usage: 1.8+ MB


Look at `df.info()` carefully. Note which columns pandas
thinks are `object` — that means text. The date column will be one of them,
which is the whole problem.


## 3. Record where you are starting

Before changing anything, capture the numbers. At the end you will compare
against these and prove the cleaning worked.


In [12]:
before = {
    'rows':       len(df),
    'duplicates': len(df) - df['result_id'].nunique(),
    'missing':    df.isna().sum().sum(),
    'categories': df['term'].nunique(),
}
before


{'rows': 30120, 'duplicates': 120, 'missing': np.int64(1507), 'categories': 12}

---
## 4. Remove duplicate rows

Week 2 told you how many exact duplicates there are. `drop_duplicates()`
removes them, keeping the first occurrence.


In [13]:
print("before:", len(df))
df = df.drop_duplicates()
print("after: ", len(df))
print("removed:", before['rows'] - len(df))


before: 30120
after:  30000
removed: 120


**TODO — write down the number removed. Does it match your
week 2 figure?**

If it does not, you are looking at something different from what you counted.
Work out which before continuing.


---
## 5. Standardise the messy categories

This is the one that would silently split your totals in Power BI. `term`
has the same values written several ways — different capitalisation, stray
spaces.

`.str.strip()` removes leading and trailing spaces. `.str.title()` makes it
Title Case. Pick one form and apply it everywhere.


In [14]:
# what it looks like now
df['term'].value_counts(dropna=False)


,count
term,
Term 2,8886
Term 3,8778
Term 1,8686
TERM 2,441
TERM 3,427
term 1,405
Term 2,404
Term 1,402
TERM 1,401


In [15]:
df['term'] = df['term'].str.strip().str.title()

df['term'].value_counts(dropna=False)


,count
term,
Term 2,10128
Term 3,9978
Term 1,9894


**TODO — how many categories now, and how many before?**

Now do the same for the other text columns. Week 2 should have told you which
ones are affected — it is not only this one.


In [16]:
# TODO: check and clean the text columns in your dimension tables
students['sex'] = students['sex'].str.strip().str.title()
students['region'] = students['region'].str.strip().str.title()

# check your work
students['sex'].value_counts(dropna=False).head(15)


,count
sex,
Female,615
Male,585


---
## 6. Parse the dates

`exam_date` is text, in four different formats. `pd.to_datetime` with
`format='mixed'` handles them, and `dayfirst=True` tells it to read `03/04/2025`
as 3 April rather than 4 March.

**This is a real decision, not a setting.** Nothing in the data proves which
reading is right. Whatever you choose, write it down in your cleaning notes and
be ready to defend it.


In [17]:
# what formats are present
df['exam_date'].str.len().value_counts()


,count
exam_date,
10,28179
11,1821


In [18]:
df['exam_date'] = pd.to_datetime(
    df['exam_date'],
    format='mixed',
    dayfirst=True,
    errors='coerce'      # anything unparseable becomes NaT rather than crashing
)

print("could not parse:", df['exam_date'].isna().sum())
print("range:", df['exam_date'].min(), "to", df['exam_date'].max())

could not parse: 0
range: 2024-01-01 00:00:00 to 2025-12-30 00:00:00


**TODO — does that date range make sense now?**

Compare it with what the text version gave you in week 2. This is where the
text-sorting problem finally goes away.

If any rows failed to parse, decide what to do with them and say why.


---
## 7. Deal with impossible values

Week 2 found scores above the maximum of 100. Look at them before deciding.


In [19]:
bad = df[df['score'] > 100]
print("rows affected:", len(bad))
bad.head(10)


rows affected: 55


,result_id,exam_date,student_id,subject_id,teacher_id,term,score,attendance_pct
55,11577,2024-08-09,664,9,42,Term 3,115.0,97.5
848,26658,2024-04-18,656,4,28,Term 1,109.4,92.7
2025,19904,2024-07-12,1046,3,41,Term 2,121.9,94.5
2471,19812,2024-05-09,265,5,36,Term 3,107.6,82.6
3249,26770,2024-09-26,179,7,15,Term 1,110.2,90.9
3490,16163,2024-01-17,211,6,26,Term 3,108.6,79.5
3719,23050,2025-05-08,770,1,23,Term 2,103.6,86.1
3886,21191,2025-11-18,193,10,46,Term 2,124.5,97.7
4624,28424,2024-01-07,995,2,14,Term 2,127.5,79.7
5129,29516,2024-08-10,284,1,2,Term 2,139.4,93.1


**TODO — decide, and write down why.**

Three defensible options. There is no single right answer, but there is a wrong
one: doing it silently.

1. **Drop them.** Clean, but you lose whatever else was in those rows.
2. **Set them to NULL.** Keeps the row, marks the value as unknown.
3. **Fix them** — if a negative looks like a data-entry sign error, taking the
   absolute value may be justified. Only if you can argue it.


In [20]:
# TODO: implement your decision. One of these, or your own.

# option 1 — drop
# df = df[~df.index.isin(bad.index)]

# option 2 — set to NULL
# df.loc[bad.index, 'COLUMN'] = np.nan

print("rows now:", len(df))


rows now: 30000


---
## 8. Deal with orphan keys

Some `student_id` values in your fact table point at
`students` records that do not exist. A plain join would drop these rows
silently — which is exactly why you are handling them deliberately.


In [21]:
valid = set(students['student_id'])
orphans = df[~df['student_id'].isin(valid)]

print("orphan rows:", len(orphans))
orphans[['result_id', 'student_id']].head(10)


orphan rows: 75


,result_id,student_id
140,9324,90027
613,12630,90016
622,17627,90063
1045,9426,90050
1099,10385,90037
1763,3365,90031
1855,11930,90013
2461,4998,90024
2547,26005,90067
3388,2942,90005


**TODO — decide, and write down why.**

1. **Drop them.** Simple, and you lose real transactions.
2. **Keep them, pointing at an "Unknown" record.** Preserves the totals, and
   your dashboard shows an Unknown category — which is honest.

Check every foreign key, not just this one.


In [22]:
# TODO: implement your decision

# option 1 — drop
# df = df[df['student_id'].isin(valid)]

# option 2 — add an Unknown row to the dimension, then repoint orphans at it
# unknown = pd.DataFrame([{'student_id': -1}])
# students = pd.concat([students, unknown], ignore_index=True)
# df.loc[~df['student_id'].isin(valid), 'student_id'] = -1

print("rows now:", len(df))


rows now: 30000


---
## 9. Handle missing values

Decide **per column**. A NULL is not always a mistake — sometimes it means
something real, and filling it in would be inventing data.


In [23]:
df.isna().sum().sort_values(ascending=False)


,0
score,1500
result_id,0
student_id,0
exam_date,0
subject_id,0
teacher_id,0
term,0
attendance_pct,0


**TODO — for each column with missing values, decide and record:**

| Column | How many | Decision | Why |
|---|---|---|---|
| `score` | | | |
| | | | |

Options: leave as NULL (honest, Power BI shows blanks), fill with a label like
`'Unknown'` (good for text you will group by), or drop the row (only if the row
is useless without it).


In [24]:
# TODO: implement your decisions

# example — label missing text so it groups properly in Power BI
# df['score'] = df['score'].fillna('Unknown')

df.isna().sum().sort_values(ascending=False).head()


,0
score,1500
result_id,0
student_id,0
exam_date,0
subject_id,0


---
## 10. Prove it worked

Re-run your week 2 checks on the cleaned data. Every problem should now be
gone or accounted for.


In [25]:
after = {
    'rows':       len(df),
    'duplicates': len(df) - df['result_id'].nunique(),
    'missing':    df.isna().sum().sum(),
    'categories': df['term'].nunique(),
}

pd.DataFrame([before, after], index=['before', 'after'])


,rows,duplicates,missing,categories
before,30120,120,1507,12
after,30000,0,1500,3


**TODO — explain every number that changed.**

If rows went down, you should be able to say exactly how many were duplicates,
how many were impossible values, and how many were orphans. If the numbers do
not add up, something happened that you did not intend.


---
## 11. Write the clean tables back

Into **your own schema**, not the raw one. `if_exists='replace'` rebuilds the
table each time you run the notebook — use `'append'` by mistake and running
twice silently doubles your data.


In [28]:
df.to_sql('results_clean', engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False)
students.to_sql('students_clean', engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False)
subjects.to_sql('subjects_clean', engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False)
teachers.to_sql('teachers_clean', engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False)

print("written")


written


### Confirm they landed


In [29]:
pd.read_sql(f"""
    SELECT relname AS table_name, reltuples AS rows
    FROM pg_class c
    JOIN pg_namespace n ON n.oid = c.relnamespace
    WHERE n.nspname = '{SCHEMA_MINE}' AND relkind = 'r'
    ORDER BY relname
""", engine)


,table_name,rows
0,results_clean,-1.0
1,students_clean,-1.0
2,subjects_clean,-1.0
3,teachers_clean,-1.0


---
## Before you finish week 3

- [ ] This notebook runs top to bottom without errors, from a fresh runtime
- [ ] Someone else in the group has run it and got the same tables
- [ ] Every TODO above has a written answer
- [ ] Your cleaning decisions and reasons are in your week 3 form
- [ ] This notebook is committed to `notebooks/` in your repository
- [ ] Your password is **not** anywhere in the notebook

**Test it properly:** Runtime → Restart runtime, then Run all. If it fails, it
is not finished.

Next week you connect Power BI to `group3b` and build the model on these tables.
